In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.transform import Rotation
# Edits
import matplotlib as mpl
from mpl_toolkits.mplot3d import Axes3D

from scipy.integrate import odeint
from scipy.optimize import fsolve
import scipy.constants as cts
import pylcp
import pylcp.atom as atom
from pylcp.fields import conventional3DMOTBeams as conventional3DMOTBeams2
from pylcp.common import bisectFindChangeValue, progressBar
#plt.style.use('paper')

In [ ]:
class gaussianBeam(pylcp.laserBeam):
    def __init__(self, kvec=np.array([1,0,0]), pol=np.array([0,0,1]), beta=1., delta=0., wb=1., **kwargs):
        if callable(kvec):
            raise TypeError('kvec cannot be a function for a Gaussian beam.')

        if callable(pol):
            raise TypeError('Polarization cannot be a function for a Gaussian beam.')

        # Use super class to define kvec(R, t), pol(R, t), and delta(t)
        super().__init__(kvec=kvec, pol=pol, delta=delta, **kwargs)

        # Save the constant values (might be useful):
        self.con_kvec = kvec
        self.con_khat = kvec/np.linalg.norm(kvec)
        self.con_pol = self.pol(np.array([0., 0., 0.]), 0.)

        # Save the parameters specific to the Gaussian beam:
        self.beta_max = beta # central saturation parameter
        self.wb = wb # 1/e^2 radius
        self.define_rotation_matrix()

    def define_rotation_matrix(self):
        # Angles of rotation:
        th = np.arccos(self.con_khat[2])
        phi = np.arctan2(self.con_khat[1], self.con_khat[0])
        
        # Use scipy to define the rotation matrix
        self.rmat = Rotation.from_euler('ZY', [phi, th]).inv().as_matrix()

    def beta(self, R=np.array([0., 0., 0.]), t=0.):
        # Rotate up to the z-axis where we can apply formulas:
        Rp = np.einsum('ij,j...->i...', self.rmat, R)
        rho_sq=np.sum(Rp[:2]**2, axis=0)
        # Return the intensity:
        return self.beta_max*np.exp(-2*rho_sq/self.wb**2)

In [ ]:
k = np.array([0., 0., -1.])
k = k/np.linalg.norm(k)
laserBeam = gaussianBeam(kvec=k, wb=1., pol=1.)
laserBeam.rmat @ k

In [ ]:
x_beta = 2
X, Y = np.meshgrid(np.linspace(-x_beta, x_beta, 101),
                   np.linspace(-x_beta, x_beta, 101))
z_tests = [-1., 0, 1.] # position

plt.figure("Laser Beams", figsize=(4, 1.5*6)) # 6 beams
plt.clf()
# pr = cProfile.Profile()

for ii, z_test in enumerate(z_tests):
    Z = z_test*np.ones(X.shape)
    Rt=np.array([X, Y, Z])

    #pr.enable()
    """it = np.nditer([X, Y, Z, None])
    for (x, y, z, beta) in it:
        beta[...] = laserBeam.beta(np.array([x, y, z]), 0.)
    BETA = it.operands[3]"""
    #pr.disable()
    
    #pr.enable()
    BETA = laserBeam.beta(Rt)
    #pr.disable()

    plt.subplot(1., len(z_tests), ii+1)
    plt.imshow(BETA, origin='lower',
               extent=(-x_beta, x_beta,
                       -x_beta, x_beta))
    plt.clim((0, 1))
    plt.set_cmap('jet')
    # Make a cross-hair:
    plt.plot([0, 0], [-x_beta, x_beta],
             'w-', linewidth=0.25)
    plt.plot([-x_beta, x_beta], [0, 0],
             'w-', linewidth=0.25)